# GTM Message Sending Test

Use this notebook to test whether a browser page can load your Google Tag Manager container and push events into `window.dataLayer`.

This does **not** write rows to Unity Catalog. It is only a browser-side GTM / Tag Assistant test helper.

Recommended workflow:

1. Open GTM Preview / Tag Assistant for container `GTM-K29QPLV2`.
2. Run the setup cell below.
3. Run one of the event test cells.
4. Confirm the event appears in Tag Assistant.

In [ ]:
dbutils.widgets.text("gtm_container_id", "GTM-K29QPLV2", "GTM container ID")
dbutils.widgets.dropdown(
    "event_name",
    "add_to_cart",
    ["page_view", "view_item", "add_to_cart", "purchase", "abandon_cart", "sign_up", "account_deleted"],
    "Event name",
)
dbutils.widgets.text("test_user_id", "notebook-test-user", "Test user ID")
dbutils.widgets.text("test_email", "nikolaos.servos@live.com", "Test email")

GTM_CONTAINER_ID = dbutils.widgets.get("gtm_container_id").strip()
EVENT_NAME = dbutils.widgets.get("event_name").strip()
TEST_USER_ID = dbutils.widgets.get("test_user_id").strip()
TEST_EMAIL = dbutils.widgets.get("test_email").strip()

print(f"GTM container: {GTM_CONTAINER_ID}")
print(f"Event name:    {EVENT_NAME}")
print(f"User ID:       {TEST_USER_ID}")
print(f"Email:         {TEST_EMAIL}")

## Load GTM and Push a Configurable Event

Run this cell while GTM Tag Assistant is connected. It injects your GTM container into the notebook output iframe and pushes one event into `window.dataLayer`.

In [ ]:
import json
import uuid

cart_id = str(uuid.uuid4())
transaction_id = str(uuid.uuid4())

base_payload = {
    "event": EVENT_NAME,
    "user_id": TEST_USER_ID,
    "cart_id": cart_id,
    "email": TEST_EMAIL,
    "first_name": "GTM",
    "surname": "Tester",
}

ecommerce_payload = {
    "cart_id": cart_id,
    "currency": "EUR",
    "value": 99.99,
    "items": [
        {
            "item_id": "notebook-test-product",
            "item_name": "Notebook GTM Test Product",
            "price": 99.99,
            "currency": "EUR",
            "quantity": 1,
        }
    ],
}

if EVENT_NAME == "purchase":
    ecommerce_payload["transaction_id"] = transaction_id

if EVENT_NAME in {"view_item", "add_to_cart", "purchase", "abandon_cart"}:
    base_payload["ecommerce"] = ecommerce_payload

if EVENT_NAME == "page_view":
    base_payload.update(
        {
            "page_path": "/gtm-notebook-test",
            "page_title": "GTM Notebook Test",
            "page_location": "https://example.com/gtm-notebook-test",
        }
    )

payload_json = json.dumps(base_payload)
container_json = json.dumps(GTM_CONTAINER_ID)

displayHTML(f"""
<div style="font-family: system-ui, sans-serif; padding: 16px; border: 1px solid #ddd; border-radius: 8px;">
  <h3>GTM test event pushed</h3>
  <p><strong>Container:</strong> {GTM_CONTAINER_ID}</p>
  <p><strong>Event:</strong> {EVENT_NAME}</p>
  <pre id="gtm-payload" style="white-space: pre-wrap; background: #f6f8fa; padding: 12px; border-radius: 6px;"></pre>
</div>
<script>
(function() {{
  const containerId = {container_json};
  const payload = {payload_json};
  window.dataLayer = window.dataLayer || [];
  window.dataLayer.push({{ ecommerce: null }});
  window.dataLayer.push({{ 'gtm.start': Date.now(), event: 'gtm.js' }});

  const script = document.createElement('script');
  script.async = true;
  script.src = 'https://www.googletagmanager.com/gtm.js?id=' + encodeURIComponent(containerId);
  document.head.appendChild(script);

  window.dataLayer.push({{ ...payload, timestamp: new Date().toISOString() }});
  document.getElementById('gtm-payload').textContent = JSON.stringify(payload, null, 2);
  console.log('[gtm-notebook-test] pushed', payload, window.dataLayer);
}})();
</script>
""")

## What to Check

In GTM Tag Assistant, you should see the selected event name, for example `add_to_cart` or `purchase`.

In your browser console, you can also inspect:

```javascript
window.dataLayer
```

If the event appears in `window.dataLayer` but not in GA4 or your server-side destination, the GTM container still needs tags/triggers configured for that event.